# Fine-tune YOLOv11 on the volleyball dataset (Colab GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Input:** `volleyball-detection-split.zip` (the dataset AFTER the leak-free
temporal split made by `fyp/prepare_detector_dataset.py` — do NOT upload the
raw Roboflow zip, it has no valid/ split).

**Output:** `volleyball_best.pt` — downloaded at the end. Expected total time ≈ 20–40 min.

Notes on the data: labels are Roboflow **polygons**; Ultralytics converts them
to boxes automatically for the `detect` task. Classes are `{0: ball, 1: player}`
(NOT COCO ids) — the project pipeline resolves ids by name, so this is safe.

In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable the GPU runtime first!'

In [ ]:
# --- Get the dataset zip into Colab -------------------------------------
# OPTION A (recommended for a ~230 MB file): put volleyball-detection-split.zip
# in your Google Drive first, then set DRIVE_ZIP below.
# OPTION B: leave DRIVE_ZIP = None and use the browser upload widget (slower).
import os
DRIVE_ZIP = None   # e.g. '/content/drive/MyDrive/volleyball-detection-split.zip'

if DRIVE_ZIP:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = DRIVE_ZIP
else:
    from google.colab import files
    up = files.upload()          # pick volleyball-detection-split.zip
    zip_path = '/content/' + list(up.keys())[0]
print('zip:', zip_path, os.path.getsize(zip_path) // 1_000_000, 'MB')

In [ ]:
# --- Unzip + re-patch data.yaml path for THIS machine --------------------
import re, zipfile, os
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
DS = '/content/volleyball-detection.yolov11'
yaml_path = os.path.join(DS, 'data.yaml')
text = open(yaml_path).read()
text = re.sub(r'(?m)^path\s*:.*$', 'path: ' + DS, text)
open(yaml_path, 'w').write(text)
print(text)
for split in ('train', 'valid'):
    n = len(os.listdir(os.path.join(DS, split, 'images')))
    print(split, n, 'images')
assert os.path.isdir(os.path.join(DS, 'valid', 'images')), 'valid/ split missing!'

In [ ]:
# --- Train (same hyper-parameters as fyp/train_detector.py) --------------
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(
    data=yaml_path,
    epochs=100, imgsz=640, batch=16,
    patience=25, seed=0, cos_lr=True, plots=True,
    project='/content/runs', name='volleyball_yolo11', exist_ok=True,
)
best = str(results.save_dir) + '/weights/best.pt'
print('BEST:', best)

In [ ]:
# --- Verify: class map, name-based id resolution, val metrics ------------
from ultralytics import YOLO
m = YOLO(best)
print('class map:', m.names)
person_id = next(i for i, n in m.names.items()
                 if str(n).lower() in ('person', 'player', 'players', 'athlete'))
ball_id = next((i for i, n in m.names.items() if 'ball' in str(n).lower()), None)
print('resolved  person =', person_id, ' ball =', ball_id)
metrics = m.val(data=yaml_path, imgsz=640)
print('mAP50 =', round(metrics.box.map50, 3), ' mAP50-95 =', round(metrics.box.map, 3))
print('per-class mAP50:', dict(zip(m.names.values(), [round(x,3) for x in metrics.box.maps])))

In [ ]:
# --- Download the weights (rename for the project) ------------------------
import shutil
shutil.copy(best, '/content/volleyball_best.pt')
from google.colab import files
files.download('/content/volleyball_best.pt')
# Also grab the training curves / confusion matrix for your report:
files.download(str(results.save_dir) + '/results.png')
files.download(str(results.save_dir) + '/confusion_matrix.png')

## Back on your laptop — the exact chain (do not skip steps)

```bat
:: 0. put the downloaded weights into the project
copy volleyball_best.pt "C:\Users\KESHAV\Downloads\fyp (2)\fyp\volleyball_best.pt"

:: 1. verify locally (loads weights, checks class ids, predicts a val image)
.venv\Scripts\python fyp\train_detector.py --verify-only fyp\volleyball_best.pt

:: 2. re-extract ALL training CSVs with the SAME detector (train/serve consistency)
.venv\Scripts\python fyp\prepare_training_data.py dataset --output-dir training_csv --yolo-model fyp\volleyball_best.pt --clean-output

:: 3. retrain the Mamba on the re-extracted CSVs
.venv\Scripts\python fyp\train_mamba.py training_csv --augment --epochs 80 --checkpoint mamba_checkpoint_v2.pt

:: 4. run the full pipeline with the custom detector
.venv\Scripts\python fyp\pipeline.py "videoplayback (3).mp4" --yolo-model fyp\volleyball_best.pt --tracker botsort --auto-court
```

Why step 2 matters: the Mamba was trained on features extracted by the OLD
detector. Serving a NEW detector with an old Mamba is a silent train/serve
mismatch — positions shift, velocities shift, and the classifier sees a
distribution it never trained on.